In [4]:
import pandas as pd
import os

### PREPROCESS
  """Standardize column names"""
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.str.lower()

    ### Clean numeric columns (remove commas)
    numeric_cols = [
        'produced_material_quantity',
        'component_material_quantity'
    ]

    for col in numeric_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(',', '', regex=False)
            .astype(float)
        )

    return df


#  AGGREGATE BOM (Bill of Material)

def aggregate_bom(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate duplicate edges:
    produced_material → component_material
    """

    df_agg = (
        df.groupby([
            'produced_material',
            'component_material',
            'year',
            'plant_id'
        ])
        .agg({
            'component_material_quantity': 'sum',
            'component_material_release_type': 'first',
            'component_material_production_type': 'first'
        })
        .reset_index()
    )

    return df_agg


### BUILD LOOKUPS

def build_lookups(df: pd.DataFrame, df_agg: pd.DataFrame):
    """
    - material_lookup → node attributes
    - bom_lookup → aggregated edges
    """

    ### Material attributes
    material_lookup = (
        df.groupby(['produced_material', 'year', 'plant_id'])
        .agg({
            'produced_material_release_type': 'first',
            'produced_material_production_type': 'max',
            'produced_material_quantity': 'max'
        })
        .to_dict('index')
    )

    ### Use aggregated BOM
    bom_lookup = (
        df_agg.groupby(['produced_material', 'year', 'plant_id'])
        .apply(lambda x: x.to_dict('records'), include_groups=False)
        .to_dict()
    )

    return material_lookup, bom_lookup


### EXPLOSION

def explode_bom(df: pd.DataFrame) -> pd.DataFrame:
    df_agg = aggregate_bom(df)
    material_lookup, bom_lookup = build_lookups(df, df_agg)

    results = []

    def recurse(fin_id, current_material, year, plant, visited):
        key = (current_material, year, plant)

        if key in visited:
            return
        visited.add(key)

        current_attrs = material_lookup.get(key, {})
        components = bom_lookup.get(key, [])

        for row in components:
            comp_id = row.get('component_material')
            comp_release = row.get('component_material_release_type')
            comp_prod_type = row.get('component_material_production_type')
            comp_qty = row.get('component_material_quantity')

            comp_attrs = material_lookup.get((comp_id, year, plant), {})

            results.append({

                ### DIMENSIONS

                'plant': plant,
                'year': year,


                ### FIN

                'fin_material_id': fin_id,
                'fin_material_release_type': material_lookup[(fin_id, year, plant)].get('produced_material_release_type'),
                'fin_material_production_type': material_lookup[(fin_id, year, plant)].get('produced_material_production_type'),
                'fin_production_quantity': material_lookup[(fin_id, year, plant)].get('produced_material_quantity'),


                ### PROD (CURRENT)

                'prod_material_id': current_material,
                'prod_material_release_type': current_attrs.get('produced_material_release_type'),
                'prod_material_production_type': current_attrs.get('produced_material_production_type'),
                'prod_material_production_quantity': current_attrs.get('produced_material_quantity'),


                ### COMPONENT

                'component_id': comp_id,
                'component_material_release_type': comp_release,
                'component_material_production_type': comp_prod_type or comp_attrs.get('produced_material_production_type'),
                'component_consumption_quantity': comp_qty
            })

            ### recurse only for valid expandable nodes
            if comp_release in ('PROD', 'FIN'):
                recurse(fin_id, comp_id, year, plant, visited)

        visited.remove(key)


    ### START FROM FIN ROOTS

    fin_roots = (
        df[df['produced_material_release_type'] == 'FIN']
        [['produced_material', 'year', 'plant_id']]
        .drop_duplicates()
    )

    for row in fin_roots.itertuples(index=False):
        recurse(
            fin_id=row.produced_material,
            current_material=row.produced_material,
            year=row.year,
            plant=row.plant_id,
            visited=set()
        )

    result_df = pd.DataFrame(results)

    ### normalize column names
    result_df.columns = [col.lower() for col in result_df.columns]

    return result_df

### EXECUTION

data_path = 'task_2_data.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    df = preprocess_data(df)

    print("Running BOM explosion with aggregation...")
    result_df = explode_bom(df)

    if not result_df.empty:
        print(f"Done. Rows: {len(result_df)}")
        print(result_df.head(10))

        # Optional save
        result_df.to_csv('bom_exploded_clean.csv', index=False)
    else:
        print("No results found.")
else:
    print(f"File not found: {data_path}")

Running BOM explosion with aggregation...
Done. Rows: 110
    plant  year  fin_material_id fin_material_release_type  \
0  RLT_10  2024            10000                       FIN   
1  RLT_10  2024            10000                       FIN   
2  RLT_10  2024            10000                       FIN   
3  RLT_10  2024            10000                       FIN   
4  RLT_10  2024            10000                       FIN   
5  RLT_10  2024            10000                       FIN   
6  RLT_10  2024            10000                       FIN   
7  RLT_10  2024            10000                       FIN   
8  RLT_10  2024            10000                       FIN   
9  RLT_10  2024            10000                       FIN   

   fin_material_production_type  fin_production_quantity  prod_material_id  \
0                          8002                   1057.0             10000   
1                          8002                   1057.0             50000   
2                        